In [ ]:
import pandas as pd

In [ ]:
# Process name.basics.tsv - kept only actors & actresses, and drop birth/death year since we won't be using it
data_actornames = pd.read_csv("../data/name.basics.tsv", sep="\t")
data_actornames = data_actornames.drop(columns=["deathYear", "birthYear"])
data_actornames = data_actornames[data_actornames["primaryProfession"].str.contains("actor|actress", na=False)]
data_actornames = data_actornames[data_actornames["knownForTitles"].str.contains(",")]
#data_actornames.to_csv("../data/actor_names.csv", index=False)
#data_actornames.to_parquet("../data/name.basics.parquet", engine="pyarrow", index=False)

In [ ]:
def invert_mapping(
    df: pd.DataFrame,
    id_col: str = "nconst",
    titles_col: str = "knownForTitles",
    null_value: str = "\\N",
) -> pd.DataFrame:
    result = df[[id_col, titles_col]].dropna()
    result = result[result[titles_col] != null_value]

    result = result.copy()
    result[titles_col] = result[titles_col].str.split(",")
    result = result.explode(titles_col)
    result[titles_col] = result[titles_col].str.strip()
    result = result[result[titles_col] != ""]

    movie_df = (
        result.groupby(titles_col)[id_col]
        .apply(lambda x: ",".join(sorted(x.unique())))
        .reset_index()
    )
    movie_df.columns = ["tconst", "personIds"]

    return movie_df.sort_values("tconst").reset_index(drop=True)


In [5]:
movie_list = invert_mapping(data_actornames)
movie_list = movie_list[movie_list["personIds"].str.contains(",")]

In [6]:
title_basics = pd.read_csv("../data/title.basics.tsv", sep="\t")
title_basics = title_basics[title_basics["titleType"] == "movie"]
title_basics["startYear"] = pd.to_numeric(title_basics["startYear"], errors="coerce")
title_basics = title_basics[(title_basics["startYear"] > 1957) & (title_basics["startYear"] <= 2026)]
title_basics = title_basics[~title_basics["genres"].str.contains("Documentary", na=False)]
title_basics = title_basics.drop(columns=["titleType", "isAdult", "endYear", "runtimeMinutes", "genres"])

In [21]:
merge_movielist = movie_list.merge(title_basics, on="tconst", how="inner")
#merge_movielist.to_csv("../data/movie_data.csv", index=False)
#merge_movielist.to_parquet("../data/movie_data.parquet", engine="pyarrow", index=False)